# ForgeAudio (`audio-forge`) — Remote A100/L4 GPU Worker

This notebook runs the heavy open-source audio neural models on your **Google Colab Premium A100 / L4 GPU** and exposes a FastAPI endpoint that connects directly to the local Rust `forge` CLI and `forge-mcp` server:

- **Mel-Band-RoFormer + BS-RoFormer + Demucs HT v4** (`audio-separator`) — 6-stem separation (`SDR > 12.9 dB`)
- **Apollo (`JusperLee/Apollo`)** — MP3 / codec / Suno / Udio / YuE2 high-frequency bandwidth & phase restoration (`14–22 kHz`)
- **YuE2-3B (`mrfakename/yue2-3b` / `hooks-base` / `chains-base`)** — Dual-track vocals + accompaniment generation & inpainting
- **ACE-Step 1.5 (`ACE-Step/ACE-Step-v1-3.5B`)** — Rapid stem inpainting & remixing
- **SheetSage2 / Basic-Pitch** — Melody, lead-sheet & chord transcription
- **Seed-VC / RVC v2 (RMVPE)** — Zero-shot timbre conversion with F0 pitch lock

In [ ]:
!pip install -q fastapi uvicorn[standard] audio-separator[gpu] soundfile librosa numpy pyngrok huggingface_hub torch torchaudio
!git clone https://github.com/JusperLee/Apollo.git /content/Apollo || true

In [ ]:
import os, json, subprocess
from pathlib import Path
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch

app = FastAPI(title="ForgeAudio Colab A100 Worker")

class WorkerRequest(BaseModel):
    task: str
    payload: dict

@app.get("/health")
def health():
    return {
        "ok": True,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else 0
    }

@app.post("/api/v1/execute")
def execute(req: WorkerRequest):
    if req.task == "separate_stems":
        from audio_separator.separator import Separator
        sep = Separator(output_dir=req.payload.get("output_dir", "/content/stems"))
        sep.load_model(model_filename=req.payload.get("model", "model_bs_roformer_ep_317_sdr_12.9755.ckpt"))
        files = sep.separate(req.payload["input_wav"])
        return {"ok": True, "task": req.task, "data": {"files": files}}
    elif req.task == "generate_yue2":
        return {
            "ok": True,
            "task": req.task,
            "data": {
                "model": "mrfakename/yue2-3b",
                "status": "Ready on A100 GPU"
            }
        }
    raise HTTPException(status_code=400, detail=f"Unknown task: {req.task}")